# 02_03 Byte pair encoding (BPE)

*📖 Book: section 2.5*

**How to use this notebook:** click a cell and press Shift+Enter to run just that cell.
Before each code cell, jot down what you think it will do. Then run it and see if you were right.

Run the cells in order, from the top. **A cell can use what earlier cells made**, so if you skip ahead you'll get `NameError: name '…' is not defined`. If that happens, run the cells above it, or just click Run All.

> **Where this fits**
>
> ```
> Odyssey text → [numbers] → model reads → model learns → model writes → MCP tool
> ```
>
> You are here: **numbers**
>
> Project files this connects to:
>
> - `src/homer_ai/tokenizer.py`
>
> **This notebook:** One character per token makes the text long and each token nearly meaningless. Can tokens be bigger than one character? BPE is the standard answer.
>
> For the big picture, what we're building and why, see the [course map](../00_start_here/00_00_course_map.ipynb).

## 1. What you'll learn

- Why one-letter tokens are a problem, and why whole words are too
- What BPE is, and how it builds tokens out of common letter groups
- How it chooses those groups automatically, by counting
- How a small BPE works from start to finish on Book 1

## 2. Words you'll meet

| Word | Meaning in plain English | Example |
|---|---|---|
| **window** | how many tokens the model can look at at one time | our model: 24 tokens |
| **subword** | a piece of a word: bigger than a letter, smaller than a word | `πολλ` + `ῶν` |
| **pair** | two tokens that sit next to each other in the text | in `πολ`: `πο` and `ολ` |
| **merge** | joining a pair into one new token | `λ` + `α` → `λα` |

Words you've met before: **token** ([02_00](../02_text_to_numbers/02_00_chapter_overview.ipynb)), **vocabulary** ([02_00](../02_text_to_numbers/02_00_chapter_overview.ipynb)), **BPE** ([02_00](../02_text_to_numbers/02_00_chapter_overview.ipynb)).

## 3. The idea

### ❓ What's wrong with one letter per token?

It makes the text very long. The model can only look at a fixed number of tokens at once. That's its **window**, and ours is 24 tokens. (In the code the number is called `chunk_size`, and each 24-token piece of text is a chunk.) With one token per letter, 24 tokens is only about 4 words of Homer. It's like reading a book through a tiny hole in a card.

### ❓ Is there a second problem?

Yes: single letters carry almost no meaning. `λ` on its own tells the model very little, while `πολλ` ("many") tells it a lot. With single letters, the model has to work out from scratch, every single time, that `π` + `ο` + `λ` + `λ` belong together.

### ❓ Then why not use whole words as tokens?

That would fix both problems: fewer tokens, and each one means something. But Greek changes the ending of a word depending on how it's used. "Many" turns up in Book 1 as `πολλά`, `πολλὰ`, `πολλοὶ`, `πολλὸν`, `πολλῶν` and more, and each form would be a separate token.

We counted Book 1:

| | Count |
|---|---|
| words in Book 1 | 3,175 |
| **different** words | 1,697 |
| words that appear **only once** | 1,262 |

So most words appear only once, and the model can't learn much from a word it sees one time. Worse, a word that isn't in the list can't be written at all.

### ❓ So what's in between letters and words?

Pieces of words, called **subwords**:

```
letters:   π | ο | λ | λ | ῶ | ν      6 tokens, little meaning
words:     πολλῶν                     1 token, but only if this exact form is in the list
pieces:    πολλ | ῶν                  2 tokens: "many" + a common ending
```

`πολλ` is shared by every form of "many", and the ending `ῶν` is shared by lots of other words. So the model learns each piece once and reuses it everywhere.

### ❓ So what is BPE?

A method that finds good pieces automatically. Picking them by hand would take forever, and it would be guesswork anyway.

About the name: **byte pair encoding** was invented in 1994 to make computer files smaller. A byte is the small basic unit computers store data in. The method found the pair of bytes that appeared most often and replaced it with one new symbol. That's the encoding part. Language models use the same trick on text.

### ❓ How does it choose the pieces? Isn't that random?

No, there's nothing random about it. A program counts, and the counts decide.

1. Start with a vocabulary of single letters.
2. Count every **pair** of neighbouring tokens in the whole text.
3. **Merge** the pair that appears most often into one new token, and add it to the vocabulary.
4. Go back to step 2. Stop after a number of merges you choose in advance.

In the Odyssey, the very first merge is `ν` followed by a space, because so many Greek words end in -ν.

### ❓ Can BPE still write a word it never merged, like a rare name?

Yes. The single letters are never removed from the vocabulary. Merges only add new tokens. It's like LEGO: you build bigger blocks, but all the small bricks stay in the box. So any word can still be written, in the worst case one letter at a time.

### ❓ Why does this matter for our Odyssey model?

With BPE, the model could:

- see more of the poem at once in the same 24-token window
- learn pieces that mean something, like `πολλ` and common endings, and not rebuild them from letters every time
- still write any Greek word, even one it has never seen

## 4. Step by step

Each step is one question. Read it, guess what the code will do, run it, then look at what you got.

### ❓ What does a BPE vocabulary look like after a couple of merges?

Here's a tiny made-up one, after two merges (`λα`, then `λλα`).

👉 **Predict first:** are the single letters still in it?

In [ ]:
vocab = ["π", "ο", "λ", "α", "λα", "λλα"]
vocab

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The single letters are all still there. The merges only added new tokens at the end.

</details>

### ❓ How is a word cut into tokens with that vocabulary?

Use the **biggest block** that fits. If none fits, fall back to single letters. You don't need to follow every line of this code. Just run it.

👉 **Predict first:** how will `πολα` be cut?

In [ ]:
def split_with_blocks(word, vocab):
    tokens = []
    i = 0
    while i < len(word):
        # try the longest block first, then shorter ones
        for size in range(len(word) - i, 0, -1):
            piece = word[i : i + size]
            if piece in vocab:
                tokens.append(piece)
                i += size
                break
    return tokens


split_with_blocks("πολα", vocab)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

`['π', 'ο', 'λα']`. There's no block for `πο`, so those two go in as single letters. `λα` is a block, so it's one token.

</details>

### ❓ Your turn: how will `αλλα` be cut?

👉 **Predict first**, and write your guess here: ...

In [ ]:
split_with_blocks("αλλα", vocab)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

`['α', 'λλα']`: one single letter, then the biggest block.

</details>

### ❓ How do I find all the neighbouring pairs?

`pairwise` walks along a word and gives you every pair of neighbours: the 1st and 2nd letter, then the 2nd and 3rd, and so on.

👉 **Predict first:** what pairs will `"πολλα"` give?

In [ ]:
from itertools import pairwise

word = "πολλα"
list(pairwise(word))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

That's 4 pairs from 5 letters. Each letter is paired with the one after it.

</details>

### ❓ How do I count which pair appears most?

A `Counter` counts how many times each thing appears, and `.most_common(2)` gives the 2 most frequent.

We use the tiny text `"λλα λλα λα"`.

👉 **Predict first:** which pair appears most often, and how many times?

In [ ]:
from collections import Counter

small_text = "λλα λλα λα"
pairs = Counter(pairwise(small_text))
pairs.most_common(2)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

`λ` + `α` appears 3 times, so it would be the **first merge**.

</details>

### ❓ Which pairs are most common in the real Odyssey?

`load_books` is our own function from `train_homer.py`. It gives one text per book, and we take Book 1.

👉 **Predict first:** which pair of characters do you think is the most common in Homer?

In [ ]:
from homer_ai.train_homer import BOOKS_DIR, load_books

text = load_books(BOOKS_DIR)[0]
pairs = Counter(pairwise(text))
pairs.most_common(5)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Each line is `((first character, second character), how many times)`, and `' '` is a space. `ν` followed by a space wins with 529, because so many Greek words end in **-ν**. Nobody picked it. The counting did.

</details>

## 5. Full example: a small BPE on Book 1

This puts all the steps together and learns **20 merges** from Book 1. Two things here are new:

- `merge_pair` goes through the token list and joins every place where the chosen pair appears.
- To cut new text into tokens, real BPE replays the merges in the same order it learned them.

You don't need to follow every line yet. Run it, and look at which pieces it learns.

👉 **Predict first:** after 20 merges, will Book 1 need many fewer tokens, or only a few fewer?

In [ ]:
def merge_pair(tokens, pair):
    """Join every place where `pair` appears next to each other into one token."""
    merged = []
    i = 0
    while i < len(tokens):
        if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == pair:
            merged.append(tokens[i] + tokens[i + 1])
            i += 2  # skip both parts of the pair
        else:
            merged.append(tokens[i])
            i += 1
    return merged


# 1. Start: every character is a token
tokens = list(text)
merges = []
sizes = [len(tokens)]  # how many tokens are left, after 0, 1, 2 ... merges

# 2-4. Count pairs, merge the most common one, repeat
for step in range(20):
    best_pair = Counter(pairwise(tokens)).most_common(1)[0][0]
    tokens = merge_pair(tokens, best_pair)
    merges.append(best_pair)
    sizes.append(len(tokens))
    left, right = best_pair
    print(f"merge {step + 1:2}: {left!r} + {right!r} -> {left + right!r}")

print()
print(f"Book 1: {len(text)} characters -> {len(tokens)} BPE tokens")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The first merges join common word endings with a space (`ν `, `ς `). After 20 merges, Book 1 is down from 19,701 characters to 16,288 tokens.

</details>

### ❓ Can I see how the token count fell?

We wrote down the token count after every merge (`sizes`). If we draw it, we can see what each merge was worth.

👉 **Predict first:** will every merge save about the same, or will the first merges save the most?

In [ ]:
import matplotlib.pyplot as plt

from homer_ai.studio_style import ACCENT, GOOD, MUTED, studio_axes, studio_figure

saved = sizes[0] - sizes[-1]
fig = studio_figure(
    "Tokens left after each merge",
    f"20 merges saved {saved:,} tokens ({saved / sizes[0]:.0%})",
)
ax = studio_axes(fig)
ax.plot(range(len(sizes)), sizes, color=ACCENT, linewidth=3, marker="o", markersize=6)
ax.set_ylim(sizes[-1] - 300, sizes[0] + 300)
ax.text(0.3, sizes[0] - 120, f"{sizes[0]:,}", color=MUTED, family="monospace")
ax.text(
    len(sizes) - 1, sizes[-1] - 170, f"{sizes[-1]:,}",
    color=GOOD, family="monospace", ha="right",
)
ax.set_xticks(range(0, 21, 2))
ax.set_xlabel("merges done")
ax.set_ylabel("tokens in Book 1")
plt.show()

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The line falls from 19,701 to 16,288, steeply at first and then more gently. The first merges join the most common pairs, so each one removes the most tokens. Later pairs are rarer, so each one saves less.

</details>

Now let's use the learned merges to cut the first line of the Odyssey into tokens.

In [ ]:
def bpe_encode(line, merges):
    """Cut new text into tokens: replay the merges in the order they were learned."""
    tokens = list(line)
    for pair in merges:
        tokens = merge_pair(tokens, pair)
    return tokens


first_line = text.splitlines()[0]
bpe_tokens = bpe_encode(first_line, merges)
print(first_line)
print(bpe_tokens)
print(f"{len(first_line)} characters -> {len(bpe_tokens)} BPE tokens")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The first line goes from 50 characters to 42 tokens. With only 20 merges the pieces are still small. Real models do tens of thousands.

</details>

## Questions learners ask

These are real questions that came up while we were building this project, with short answers.

### ❓ Why does the first merge contain a space?

The space is a character like any other. `ν` + space means "a word ended in ν", and so many Greek words end in -ν that it's the most common pair of all.

### ❓ How many merges should I choose?

Each merge adds one token to the vocabulary, so the model gets a little bigger each time. And too many merges on a small text just turns whole words into tokens. In our full example, 20 merges took Book 1 from 19,701 characters to 16,288 tokens. Real models do tens of thousands, on far more text.

### ❓ Does our model use BPE?

No. Our tokenizer works with single letters. This notebook shows how BPE works, and switching our model over to it would be a separate step.

## 6. Recap

- Single-character tokens are long and nearly meaningless; whole words don't suit Greek's many word forms.
- BPE finds the pieces in between automatically: count pairs, merge the most common, repeat.
- Single characters stay in the vocabulary, so any text can still be written.

**You can now:**

- do one BPE merge by hand: count the pairs, merge the most common one
- say why GPT-style models use pieces between a character and a word

**Next:** [02_04: the sliding window](02_04_sliding_window.ipynb): cutting the text into practice pieces for the model.